In [1]:
!pip -q install kagglehub[pandas-datasets]
!pip -q install pydot
!pip -q install graphviz
!pip -q install scikit-learn
!pip -q install matplotlib
!pip -q install seaborn
!pip -q install pandas
!pip -q install numpy
!pip -q install tensorflow
!pip -q install spacy
!pip install keras-tuner
!pip install pyarrow
!pip install polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 2.8 MB/s eta 0:00:00


In [ ]:
!python -m spacy download en_core_web_md

In [2]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import kagglehub
from kagglehub import KaggleDatasetAdapter
from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from collections import Counter
import spacy
import pandas as pd
from sklearn.utils import resample
from IPython.display import display, HTML
import builtins
from pathlib import Path
import keras_tuner as kt
from tqdm import tqdm
import json
import polars as pl
import os
import glob


In [4]:
from google.colab import drive
drive.mount('/content/drive')

drive_root = Path("/content/drive/MyDrive/Personal/Universidad/8° Semestre/RN/Entrega 2")
drive_root.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


# OBTENER INFORMACION

In [ ]:
# Carga del modelo de lenguaje de spaCy (en inglés, tamaño medio, incluye vectores)
nlp = spacy.load("en_core_web_md", disable=["ner", "parser", "textcat", "attribute_ruler"])

In [ ]:
def spacy_preprocess_sm(text):
    # Procesa el texto con el modelo de spaCy, dividiéndolo en tokens, detectando POS, etc.
    doc = nlp(text)

    # Lista de tokens preprocesados mediante lematización, minúsculas, y filtros
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct and token.lemma_ != "-PRON-"
    ]

    # Une los tokens en una cadena de texto preprocesado
    return " ".join(tokens)

In [ ]:
def spacy_preprocess_xxl(texts, batch_size=5000, n_threads=8):
    # Procesa una lista de textos con spaCy en lotes, aplicando lematización y filtros.
    results = []

    # Procesa los textos en lotes para mejorar la eficiencia
    for doc in enumerate(nlp.pipe(texts, batch_size=batch_size, n_process=n_threads)):
        # Lista de tokens preprocesados mediante lematización, minúsculas, y filtros
        tokens = [
            token.lemma_.lower()
            for token in doc
            if not token.is_stop and not token.is_punct and token.lemma_ != "-PRON-"
        ]

        # Agrega los tokens preprocesados a los resultados
        results.append(" ".join(tokens))
    return results

CARGA XXL

In [3]:
CHUNK_SIZE = 100_000
INPUT_FILE = "yelp_academic_dataset_review.json"
OUTPUT_DIR = "yelp_chunks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)
N_CHUNKS = (total_lines + CHUNK_SIZE - 1) // CHUNK_SIZE


def yield_chunks_ndjson(filename, chunk_size):
    with open(filename, "r", encoding="utf-8") as f:
        chunk = []
        for line in f:
            if line.strip():
                obj = json.loads(line)
                chunk.append(obj)
            if len(chunk) == chunk_size:
                yield chunk
                chunk = []
        if chunk:
            yield chunk


for i, chunk in enumerate(
    tqdm(yield_chunks_ndjson(INPUT_FILE, CHUNK_SIZE), total=N_CHUNKS)
):
    df = pl.DataFrame(
        {
            "text": [row.get("text", "") for row in chunk],
            "stars": [row.get("stars", None) for row in chunk],
        }
    )
    df = df.filter(pl.col("text").str.strip_chars() != "")
    texts = df["text"].to_list()

    processed_texts = spacy_preprocess_xxl(texts, batch_size=1000, n_threads=6)

    df = df.with_columns([pl.Series("text", processed_texts)])
    df = df.filter(pl.col("text").str.strip_chars() != "")
    df = df.with_columns(
        [
            pl.col("text")
            .map_elements(lambda txt: len(txt.split()), return_dtype=pl.Int32)
            .alias("token_count")
        ]
    )
    output_path = os.path.join(OUTPUT_DIR, f"chunk_{i:04d}.parquet")
    df.write_parquet(output_path)
    tqdm.write(f"Chunk {i} guardado en: {output_path}")

FileNotFoundError: [Errno 2] No such file or directory: 'yelp_academic_dataset_review.json'

In [ ]:
files = sorted(glob.glob("yelp_chunks/chunk_*.parquet"))
dfs = [pl.read_parquet(f) for f in files]
df_total = pl.concat(dfs)
df_total.write_parquet("yelp_reviews_xxl.parquet")

CARGA SM

In [ ]:
# Descarga y carga el dataset de reseñas de Yelp desde Kaggle usando kagglehub
kagglehub.dataset_download('omkarsabnis/yelp-reviews-dataset')
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "omkarsabnis/yelp-reviews-dataset",
    "yelp.csv",
)

# Conservamos únicamente las columnas necesarias: el texto de la reseña y su puntuación
df = df[['text', 'stars']]

# Se convierte a string explícitamente y se aplica lematización + limpieza a cada reseña
df['text'] = df['text'].astype(str).apply(spacy_preprocess_sm)
df = df[df['text'].str.strip() != ""]

# Se añade una nueva columna con el número de tokens (palabras) por reseña
df['token_count'] = df['text'].astype(str).apply(lambda txt: len(txt.split()))

# Se guarda el DataFrame limpio y procesado en formato Parquet para uso posterior eficiente
df.to_parquet('yelp_reviews_preprocessed.parquet')

PREPARAR DATA

In [5]:
def prepare_data(cfg, parquet_path="yelp_reviews_preprocessed.parquet", n_rows=None):
    # Cargar DF ya preprocesado
    # Se carga un DataFrame preprocesado guardado en formato Parquet.
    # Este archivo debe contener las columnas 'text', 'stars' y 'token_count'.
    if n_rows is not None:
        df = pd.read_parquet(drive_root / parquet_path).sample(n=n_rows, random_state=42)
    else:
        df = pd.read_parquet(drive_root / parquet_path)

    df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True)

    df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True)

    # Etiquetado
    if cfg["mode"] == "binary":
        # Si el modo es binario, se agrupan las clases:
        # estrellas 1-3 → 0 (negativas), estrellas 4-5 → 1 (positivas)
        df = df[df["stars"].isin([1, 2, 3, 4, 5])].copy()
        df["label"] = df["stars"].apply(lambda s: 0 if s <= 3 else 1)
        num_classes = 1  # solo una clase de salida (sigmoid)
    else:
        # Si el modo es multiclase, se asigna un valor 0-4 a cada estrella
        df = df.copy()
        df["label"] = df["stars"] - 1
        num_classes = 5  # 5 clases de salida (softmax)

    # Balanceo de clases
    if cfg["balance"]:
        # Se calcula el número objetivo de muestras por clase (120% del mínimo)
        min_c = df["label"].value_counts().min()
        target = int(min_c * 1.2)

        dfs = []
        for lab in df["label"].unique():
            # Para cada clase, se genera un DataFrame filtrado
            cls_df = df[df["label"] == lab]

            if len(cls_df) > target:
                # Si la clase tiene más muestras que el objetivo, se submuestrea
                cls_df = resample(
                    cls_df,
                    replace=False,  # sin reemplazo para submuestreo
                    n_samples=target,
                    random_state=42
                )

            dfs.append(cls_df)
        # Se combinan todas las clases y se reordenan aleatoriamente
        df = pd.concat(dfs).sample(frac=1, random_state=42)

    # División en train / validation / test
    # Primero se separa un 10% para test
    x_temp, x_test, y_temp, y_test = train_test_split(
        df["text"], df["label"],
        test_size=0.10,
        random_state=42
    )

    # Luego se separa un 10% de lo restante para validación
    x_train, x_val, y_train, y_val = train_test_split(
        x_temp, y_temp,
        test_size=0.10,
        random_state=42
    )

    # Conversión a tf.data.Dataset
    # Se crean datasets de TensorFlow a partir de los textos y etiquetas
    train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
    val_ds   = tf.data.Dataset.from_tensor_slices((x_val, y_val))
    test_ds  = tf.data.Dataset.from_tensor_slices((x_test, y_test))

    # Cálculo de class_weight (opcional)
    if cfg["use_weights"]:
        # Se calcula un peso inversamente proporcional a la frecuencia de cada clase
        w = class_weight.compute_class_weight(
            class_weight='balanced',
            classes=np.unique(y_train),
            y=np.array(y_train)
        )
        cw = dict(enumerate(w))  # se convierte a diccionario para pasarlo a model.fit
    else:
        cw = None

    # Retorno
    # Se retorna: datasets (train, val, test), los pesos, el número de clases
    # y la longitud máxima (percentil 85) para truncamiento posterior
    return train_ds, val_ds, test_ds, cw, num_classes, int(round(np.percentile(df['token_count'], 85))), df['text']

In [6]:
def format_class(k, mode):
    if mode == "binary":
        return "Negativa (0)" if k == 0 else "Positiva (1)"
    else:
        return f"{'★' * (k + 1)}{'☆' * (4 - k)} ({k})"

def print_class_distribution(counter, label, mode):
    print(f"{label}:")
    for k in sorted(counter.keys()):
        print(f"  Clase {k} – {format_class(k, mode)}: {counter[k]} ejemplos")

def sanity_check_prepare_data(train_ds, val_ds, test_ds, cw, num_classes, max_len, features):
    print("✅ Iniciando verificación de datos...\n")

    # Verificación de columnas mínimas
    expected_cols = {"text", "stars", "token_count"}
    parquet_cols  = pd.read_parquet(drive_root / "yelp_reviews_preprocessed.parquet").columns
    assert expected_cols.issubset(parquet_cols), (
        f"❌ Faltan columnas requeridas: {expected_cols - set(parquet_cols)}"
    )

    # Extraer etiquetas de cada dataset
    y_train = [y.numpy() for _, y in train_ds]
    y_val   = [y.numpy() for _, y in val_ds]
    y_test  = [y.numpy() for _, y in test_ds]

    # Distribución de clases
    c_train = Counter(y_train)
    c_val   = Counter(y_val)
    c_test  = Counter(y_test)
    c_total = c_train + c_val + c_test

    # Distribución de clases con descripción
    print("📊 Distribución de etiquetas:")
    print_class_distribution(c_train, "🟩 Train", mode="binary" if num_classes == 1 else "multiclass")
    print_class_distribution(c_val,   "🟨 Val",   mode="binary" if num_classes == 1 else "multiclass")
    print_class_distribution(c_test,  "🟥 Test",  mode="binary" if num_classes == 1 else "multiclass")
    print_class_distribution(c_total, "🧮 Total", mode="binary" if num_classes == 1 else "multiclass")

    # Tamaños de cada partición
    print(f"📦 Tamaños → train={len(y_train)}, val={len(y_val)}, test={len(y_test)}\n")

    # Pesos de clase
    if cw:
        print(f"⚖️  Pesos de clase: {cw}")
        expected_weights = 2 if num_classes == 1 else num_classes
        assert len(cw) == expected_weights, (
            f"❌ class_weight tiene {len(cw)} pesos, se esperaban {expected_weights} (modo {'binario' if num_classes == 1 else 'multiclase'})"
        )
        print()

    # Longitud observada vs. max_len seleccionado
    max_observed = builtins.max(len(str(txt).split()) for txt in features)
    print(f"🔠 Máxima longitud observada: {max_observed} tokens")
    print(f"✂️  Longitud usada (percentil 85): {max_len}\n")
    assert max_len <= max_observed, "❌ max_len supera la longitud máxima observada"

    # Ejemplo de muestra
    try:
        sample_text, sample_label = next(iter(train_ds.take(1)))
        print("📌 Ejemplo de texto:")
        print("Texto   :", sample_text.numpy()[:80])
        print("Etiqueta:", sample_label.numpy(), "\n")
    except Exception as e:
        raise RuntimeError("❌ No se pudo obtener un ejemplo del dataset.") from e

    print("✅ Todas las verificaciones pasaron correctamente.")

# MODELO

In [7]:
VOCAB_SIZE = 30000  # Tamaño del vocabulario para la vectorización

def vectorize_datasets(train_ds, val_ds, test_ds, features, max_length, vocab_size=VOCAB_SIZE):
    # 1. Crea la capa de vectorización
    vectorize_layer = tf.keras.layers.TextVectorization(
        standardize=None,
        max_tokens=vocab_size,
        output_mode='int',
        output_sequence_length=max_length
    )

    # 2. Adapta la capa al vocabulario del corpus completo
    vectorize_layer.adapt(features)

    # 3. Define la función de mapeo para cada elemento del dataset
    def vectorized_text(text: tf.Tensor, label: tf.Tensor):
        return vectorize_layer(text), label

    # 4. Aplica la vectorización a cada conjunto
    train_ds_vec = train_ds.map(vectorized_text)
    val_ds_vec   = val_ds.map(vectorized_text)
    test_ds_vec  = test_ds.map(vectorized_text)

    return train_ds_vec, val_ds_vec, test_ds_vec, vectorize_layer, len(vectorize_layer.get_vocabulary())

In [8]:
def plot_training(history, save_path: Path = None):
    if hasattr(history, "history"):
        history = history.history
    else:
        history = history

    plt.figure(figsize=(12, 4))

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history['loss'], label='Pérdida de entrenamiento')
    plt.plot(history['val_loss'], label='Pérdida de validación')
    plt.title('Evolución de la Pérdida')
    plt.xlabel('Época')
    plt.ylabel('Pérdida')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['accuracy'], label='Exactitud de entrenamiento')
    plt.plot(history['val_accuracy'], label='Exactitud de validación')
    plt.title('Evolución de la Exactitud')
    plt.xlabel('Época')
    plt.ylabel('Exactitud')
    plt.legend()

    plt.tight_layout()

    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path)
        print(f"📊 Figura guardada en: {save_path}")

    plt.show()

In [9]:
def show_classification_report(base_path):
    df = pd.read_csv(f"{base_path}/classification_report.csv")
    base_path = Path(base_path)

    rename_dict = {
        'clase': 'Class', 'Clase': 'Class', 'class': 'Class',
        'precision': 'Precision', 'Precisión': 'Precision',
        'recall': 'Recall', 'Exhaustividad': 'Recall',
        'f1-score': 'F1-score', 'F1-score': 'F1-score',
        'support': 'Support', 'Soporte': 'Support',
        'accuracy': 'Accuracy', 'Exactitud': 'Accuracy'
    }
    rename_dict = {k: v for k, v in rename_dict.items() if k in df.columns}
    df = df.rename(columns=rename_dict)

    for col in df.columns:
        if df[col].dtype == float:
            df[col] = df[col].round(3)

    fig, ax = plt.subplots(figsize=(min(12, 2+len(df.columns)), 0.5*len(df)+1))
    ax.axis('off')
    tbl = ax.table(cellText=df.values, colLabels=df.columns, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(12)
    tbl.auto_set_column_width(col=list(range(len(df.columns))))
    fig.tight_layout()
    plt.savefig(str(base_path / "classification_report.png"), bbox_inches='tight', dpi=200, transparent=True)
    plt.close()
    display(HTML(df.to_html(index=False)))

    cm_path = base_path / "confusion_matrix.csv"
    if cm_path.exists():
        cm_df = pd.read_csv(cm_path, index_col=0)
        cm = cm_df.values

        fig2, ax2 = plt.subplots(figsize=(4, 3))
        im = ax2.imshow(cm, cmap='Blues')
        ax2.set_title('Confusion Matrix')
        ax2.set_xlabel('Predicted label')
        ax2.set_ylabel('True label')

        ax2.set_xticks(np.arange(len(cm_df.columns)))
        ax2.set_xticklabels([str(col).replace('pred_', '') for col in cm_df.columns])
        ax2.set_yticks(np.arange(len(cm_df.index)))
        ax2.set_yticklabels([str(idx).replace('true_', '') for idx in cm_df.index])

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax2.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")
        plt.colorbar(im, ax=ax2)
        fig2.tight_layout()
        plt.savefig(str(base_path / "confusion_matrix.png"), bbox_inches='tight', dpi=200, transparent=True)
        plt.close()
        display(cm_df)
    else:
        print("No confusion_matrix.csv found in this folder.")

    return df

In [10]:
def evaluate_from_predictions(y_true, y_pred, num_classes, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    average_type = "binary" if num_classes == 1 else "macro"

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average=average_type, zero_division=0)
    rec  = recall_score(y_true, y_pred, average=average_type, zero_division=0)
    f1   = f1_score(y_true, y_pred, average=average_type, zero_division=0)
    cm   = confusion_matrix(y_true, y_pred)

    report_dict = classification_report(y_true, y_pred, digits=4, output_dict=True)
    pd.DataFrame(report_dict).T.to_csv(
        output_dir / "classification_report.csv",
        index_label="clase",
        encoding="utf-8"
    )

    pd.DataFrame([{
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1
    }]).to_csv(
        output_dir / "metrics_summary.csv",
        index=False,
        encoding="utf-8"
    )

    pd.DataFrame(cm).to_csv(
        output_dir / "confusion_matrix.csv",
        header=[f"pred_{i}" for i in range(cm.shape[1])],
        index=[f"true_{i}" for i in range(cm.shape[0])],
        encoding="utf-8"
    )

    show_classification_report(output_dir)

# RNN

Clasificacion Multiclase

In [ ]:
def build_model_rnn_multiclass(max_length, num_classes, vocab_size):
    def build(hp: kt.HyperParameters):
        # Entrada
        inputs = tf.keras.Input(
            shape=(max_length,), dtype=tf.int64, name="input_tokens"
        )

        # Embedding
        x = tf.keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=hp.Choice("embedding_dim", [32, 64, 128, 256]),
            embeddings_initializer=hp.Choice(
                "emb_init", ["glorot_uniform", "he_normal", "lecun_normal"]
            ),
            name="embedding"
        )(inputs)

        # SpatialDropout (opcional)
        if hp.Boolean("use_spatial_dropout"):
            x = tf.keras.layers.SpatialDropout1D(
                rate=hp.Float("spatial_dropout_rate", 0.1, 0.5, step=0.1)
            )(x)

        # Bloque RNN dinámico
        num_layers = hp.Int("num_rnn_layers", 1, 5)
        for i in range(num_layers):
            x = tf.keras.layers.SimpleRNN(
                units=hp.Int(f"rnn_units_{i}", 32, 256, step=32),
                return_sequences=(i < num_layers - 1),
                recurrent_dropout=hp.Float(f"rnn_rec_dropout_{i}", 0.0, 0.5, step=0.1),
                kernel_initializer=hp.Choice(
                    f"kernel_init_{i}", ["glorot_uniform", "he_uniform", "orthogonal"]
                ),
                recurrent_initializer=hp.Choice(
                    f"recurrent_init_{i}", ["orthogonal", "glorot_uniform"]
                ),
                kernel_regularizer=(
                    tf.keras.regularizers.L1L2(
                        l1=hp.Float(f"l1_{i}", 1e-7, 1e-4, sampling="log"),
                        l2=hp.Float(f"l2_{i}", 1e-7, 1e-3, sampling="log")
                    ) if hp.Boolean(f"use_reg_{i}") else None
                ),
                name=f"simple_rnn_{i}"
            )(x)

            if hp.Boolean(f"use_layernorm_{i}"):
                x = tf.keras.layers.LayerNormalization(name=f"layernorm_{i}")(x)

        # Densa intermedia (opcional)
        if hp.Boolean("use_intermediate_dense"):
            x = tf.keras.layers.Dense(
                units=hp.Int("dense_units", 16, 128, step=16),
                activation=hp.Choice("dense_activation", ["relu", "tanh", "gelu"]),
                kernel_regularizer=(
                    tf.keras.regularizers.l2(
                        hp.Float("dense_l2", 1e-6, 1e-2, sampling="log")
                    ) if hp.Boolean("dense_use_l2") else None
                ),
                name="dense_intermediate"
            )(x)

            if hp.Boolean("use_batchnorm"):
                x = tf.keras.layers.BatchNormalization(name="batchnorm")(x)

            if hp.Boolean("use_dense_dropout"):
                x = tf.keras.layers.Dropout(
                    rate=hp.Float("dense_dropout_rate", 0.1, 0.6, step=0.1),
                    name="dense_dropout"
                )(x)

        # Capa de salida
        outputs = tf.keras.layers.Dense(
            num_classes, activation="softmax", name="output_probs"
        )(x)

        model = tf.keras.Model(
            inputs=inputs, outputs=outputs, name="tunable_rnn_multiclass"
        )

        # Optimizador
        opt_choice = hp.Choice("optimizer", ["adam", "sgd", "rmsprop"])
        if opt_choice == "adam":
            optimizer = tf.keras.optimizers.Adam(
                learning_rate=hp.Float("adam_lr", 1e-5, 1e-3, sampling="log"),
                beta_1=hp.Float("adam_b1", 0.8, 0.99, step=0.01),
                beta_2=hp.Float("adam_b2", 0.9, 0.999, step=0.01),
                epsilon=1e-7
            )
        elif opt_choice == "sgd":
            optimizer = tf.keras.optimizers.SGD(
                learning_rate=hp.Float("sgd_lr", 1e-4, 1e-1, sampling="log"),
                momentum=hp.Float("sgd_momentum", 0.0, 0.95, step=0.05),
                nesterov=hp.Boolean("sgd_nesterov")
            )
        else:
            optimizer = tf.keras.optimizers.RMSprop(
                learning_rate=hp.Float("rms_lr", 1e-5, 1e-3, sampling="log"),
                rho=hp.Float("rms_rho", 0.8, 0.99, step=0.01),
                momentum=hp.Float("rms_momentum", 0.0, 0.9, step=0.05),
                epsilon=1e-7
            )

        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        return model

    return build

Clasificacion Binaria

In [ ]:
def build_model_rnn_binary(max_length, vocab_size):
    def build(hp: kt.HyperParameters):
        # Entrada
        inputs = tf.keras.Input(
            shape=(max_length,),
            dtype=tf.int64,
            name="input_tokens"
        )

        # Embedding
        x = tf.keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=hp.Choice("embedding_dim", [32, 64, 128, 256]),
            name="embedding"
        )(inputs)

        # SpatialDropout (opcional)
        if hp.Boolean("use_spatial_dropout"):
            x = tf.keras.layers.SpatialDropout1D(
                rate=hp.Float("spatial_dropout_rate", 0.1, 0.5, step=0.1)
            )(x)

        # Bloque RNN dinámico
        num_layers = hp.Int("num_rnn_layers", 1, 5)
        for i in range(num_layers):
            x = tf.keras.layers.SimpleRNN(
                units=hp.Int(f"rnn_units_{i}", 32, 256, step=32),
                return_sequences=(i < num_layers - 1),
                recurrent_dropout=hp.Float(f"rnn_rec_dropout_{i}", 0.0, 0.5, step=0.1),
                kernel_regularizer=(
                    tf.keras.regularizers.l2(
                        hp.Float(f"l2_{i}", 1e-6, 1e-3, sampling="log")
                    ) if hp.Boolean(f"use_reg_{i}") else None
                ),
                name=f"simple_rnn_{i}"
            )(x)

            if hp.Boolean(f"use_layernorm_{i}"):
                x = tf.keras.layers.LayerNormalization(name=f"layernorm_{i}")(x)

        # Densa intermedia (opcional)
        if hp.Boolean("use_intermediate_dense"):
            x = tf.keras.layers.Dense(
                units=hp.Int("dense_units", 16, 128, step=16),
                activation=hp.Choice("dense_activation", ["relu", "tanh", "gelu"]),
                kernel_regularizer=(
                    tf.keras.regularizers.l2(
                        hp.Float("dense_l2", 1e-6, 1e-3, sampling="log")
                    ) if hp.Boolean("dense_use_l2") else None
                ),
                name="dense_intermediate"
            )(x)

            if hp.Boolean("use_batchnorm"):
                x = tf.keras.layers.BatchNormalization(name="batchnorm")(x)

            if hp.Boolean("use_dense_dropout"):
                x = tf.keras.layers.Dropout(
                    rate=hp.Float("dense_dropout_rate", 0.1, 0.6, step=0.1),
                    name="dense_dropout"
                )(x)

        # Salida
        outputs = tf.keras.layers.Dense(
            1, activation="sigmoid", name="output_prob"
        )(x)

        model = tf.keras.Model(inputs=inputs, outputs=outputs, name="tunable_rnn_binary")

        # Optimizador
        opt_choice = hp.Choice("optimizer", ["adam", "rmsprop"])
        if opt_choice == "adam":
            optimizer = tf.keras.optimizers.Adam(
                learning_rate=hp.Float("adam_lr", 1e-5, 1e-3, sampling="log"),
            )
        else:
            optimizer = tf.keras.optimizers.RMSprop(
                learning_rate=hp.Float("rms_lr", 1e-5, 1e-3, sampling="log"),
            )

        model.compile(
            optimizer=optimizer,
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )
        return model

    return build

In [ ]:
def get_builder(num_classes, max_length, vocab_size):
    # Si el número de clases es 1, estamos haciendo clasificación binaria,
    # por lo tanto, se retorna la función que construye un modelo binario.
    # En caso contrario, se asume una clasificación multiclase (más de 2 clases),
    # y se retorna la función correspondiente para ese tipo de modelo.
    return build_model_rnn_binary(max_length, vocab_size) if num_classes == 1 else build_model_rnn_multiclass(max_length, num_classes, vocab_size)

In [ ]:
RNN_EXPERIMENTS = [
    # dict(name="rnn_bin_raw",        mode="binary",     balance=False, use_weights=False),
    # dict(name="rnn_bin_balanced",   mode="binary",     balance=True,  use_weights=False),
    # dict(name="rnn_bin_weighted",   mode="binary",     balance=False, use_weights=True),

    dict(name="rnn_multi_raw",      mode="multiclass", balance=False, use_weights=False),
    dict(name="rnn_multi_balanced", mode="multiclass", balance=True,  use_weights=False),
    dict(name="rnn_multi_weighted", mode="multiclass", balance=False, use_weights=True),
]

In [ ]:
# Directorio raíz para almacenar los logs y modelos del tuner
LOG_ROOT = drive_root / "tuner_logs_rnn_7"
LOG_ROOT.mkdir(exist_ok=True)

# Tamaño fijo de batch para entrenamiento y evaluación
BATCH_SIZE = 16

# Iteración sobre cada configuración experimental definida en RNN_EXPERIMENTS
for cfg in tqdm(RNN_EXPERIMENTS, desc="Configuraciones RNN", unit="cfg"):
    print(f"\n🔎  Buscando hiperparámetros – {cfg['name']}")

    #Preparación del conjunto de datos según la configuración experimental
    train_ds, val_ds, test_ds, cw, num_classes, max_len, features = prepare_data(cfg, "yelp_reviews_xxl.parquet", 100_000)

    #Verificación de consistencia de los datos preparados
    # sanity_check_prepare_data(train_ds, val_ds, test_ds, cw, num_classes, max_len, features)
    print()

    #Vectorización del texto (tokenización + mapeo a enteros)
    train_ds, val_ds, test_ds, vectorize_layer, vocab_size = vectorize_datasets(
        train_ds, val_ds, test_ds, features, max_len
    )

    # #Directorio base para el experimento actual
    exp_dir = LOG_ROOT/ cfg["name"]

    #Constructor de modelo dinámico para binario o multiclase
    builder = get_builder(num_classes, max_len, vocab_size)

    #Configuración del tuner bayesiano
    tuner = kt.BayesianOptimization(
        hypermodel=builder,
        objective=kt.Objective("val_accuracy", direction="max"),
        max_trials=30,
        num_initial_points=10,
        seed=42,
        directory=str(exp_dir),
        project_name=cfg["name"]
    )

    #Callback de parada temprana para evitar sobreentrenamiento
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    #Callback de TensorBoard para visualizar el entrenamiento
    logs = tf.keras.callbacks.TensorBoard(
        log_dir=f"{str(exp_dir)}/tb_logs"
    )

    #Ejecución del tuner
    tuner.search(
        train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE),
        validation_data=val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE),
        epochs=50,
        class_weight=cw if cw else None,
        callbacks=[early_stopping, logs],
        verbose=2
    )

    best_builder = get_builder(num_classes, max_len, vocab_size)

    try:
        best_model = tuner.get_best_models(1)[0]
        print("✅ Modelo cargado exitosamente desde el tuner.")
    except Exception as e:
        print(f"⚠️  No se pudo cargar el modelo desde el tuner: {e}")
        best_model = None

    # Verificación manual de que el modelo esté correctamente cargado
    if best_model is None or not best_model.built:
        print("🔁 Reconstruyendo el modelo desde cero con los mejores hiperparámetros...")
        best_hp = tuner.get_best_hyperparameters(1)[0]
        best_model = best_builder(best_hp)
    else:
        best_hp = tuner.get_best_hyperparameters(1)[0]

    # Evaluación final en el conjunto de prueba
    result = best_model.evaluate(
        test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE),
        verbose=1
    )

    # Obtener etiquetas reales y predichas
    y_true = np.concatenate([y.numpy() for _, y in test_ds.batch(BATCH_SIZE)], axis=0)
    y_pred_probs = best_model.predict(test_ds.batch(BATCH_SIZE))

    # Convertir probabilidades a clases
    print(num_classes)
    if num_classes == 1:
        y_pred = (y_pred_probs > 0.5).astype("int32").flatten()
        print(f"Predicciones binarias: {np.unique(y_pred)}")
    else:
        y_pred = np.argmax(y_pred_probs, axis=1)

    #Creación del directorio del modelo
    models_dir = LOG_ROOT / cfg["name"] / "models"
    models_dir.mkdir(parents=True, exist_ok=True)

    # Evaluar y guardar resultados
    evaluate_from_predictions(y_true, y_pred, num_classes, output_dir=models_dir)

    #Guardado de resultados en test
    with open(models_dir / "test_results.txt", "w") as f:
        f.write(f"Test Loss: {result[0]:.4f} — Test Accuracy: {result[1]:.4f}\n")

    #Guardado de los mejores hiperparámetros
    with (models_dir / "best_hyperparameters.json").open("w") as f:
        json.dump(best_hp.values, f, indent=2)

    #Guardado del mejor modelo
    best_model.save(str(models_dir / f"{cfg['name']}_best_model.h5"))
    best_model.save(str(models_dir / f"{cfg['name']}_best_model.keras"))

    # Reconstrucción del modelo con los mejores hiperparámetros
    best_builder = get_builder(num_classes, max_len, vocab_size)
    best_model_rebuilt = best_builder(best_hp)

    # Reentrenamiento final para obtener history
    history = best_model_rebuilt.fit(
        train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE),
        validation_data=val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE),
        epochs=50,
        class_weight=cw if cw else None,
        callbacks=[early_stopping],
        verbose=2
    )

    # Gráfico de las métricas de entrenamiento
    plot_training(history, models_dir / "training_plot.png")

Trial 14 Complete [00h 51m 57s]
val_accuracy: 0.6245555281639099

Best val_accuracy So Far: 0.6367777585983276
Total elapsed time: 21h 00m 44s

Search: Running Trial #15

Value             |Best Value So Far |Hyperparameter
256               |256               |embedding_dim
he_normal         |glorot_uniform    |emb_init
True              |False             |use_spatial_dropout
5                 |1                 |num_rnn_layers
256               |64                |rnn_units_0
0                 |0                 |rnn_rec_dropout_0
orthogonal        |he_uniform        |kernel_init_0
orthogonal        |glorot_uniform    |recurrent_init_0
True              |True              |use_reg_0
False             |True              |use_layernorm_0
False             |False             |use_intermediate_dense
adam              |adam              |optimizer
3.831e-05         |5.1617e-05        |adam_lr
0.82              |0.87              |adam_b1
0.98              |0.91              |adam_b2
32  

# LSTM

Clasificacion Multiclase

In [ ]:
def build_model_lstm_multiclass(max_length: int, num_classes: int, vocab_size: int):
    def build(hp: kt.HyperParameters):
        # Entrada
        inputs = tf.keras.Input(shape=(max_length,), dtype=tf.int64, name="input_tokens")

        # Embedding
        x = tf.keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=hp.Choice("embedding_dim", [32, 64, 128, 256]),
            embeddings_initializer=hp.Choice("emb_init",
                                            ["glorot_uniform", "he_normal", "lecun_normal"]),
            name="embedding"
        )(inputs)

        # SpatialDropout (opcional)
        if hp.Boolean("use_spatial_dropout"):
            x = tf.keras.layers.SpatialDropout1D(
                rate=hp.Float("spatial_dropout_rate", 0.1, 0.5, step=0.1)
            )(x)

        # Bloque LSTM dinámico
        num_layers = hp.Int("num_lstm_layers", 1, 5)
        for i in range(num_layers):
            x = tf.keras.layers.LSTM(
                units=hp.Int(f"lstm_units_{i}", 32, 256, step=32),
                return_sequences=(i < num_layers - 1),
                recurrent_dropout=hp.Float(f"lstm_rec_dropout_{i}", 0.0, 0.5, step=0.1),
                kernel_initializer=hp.Choice(
                    f"kernel_init_{i}", ["glorot_uniform", "he_uniform", "orthogonal"]
                ),
                recurrent_initializer=hp.Choice(
                    f"recurrent_init_{i}", ["orthogonal", "glorot_uniform"]
                ),
                kernel_regularizer=(
                    tf.keras.regularizers.L1L2(
                        l1=hp.Float(f"l1_{i}", 1e-7, 1e-4, sampling="log"),
                        l2=hp.Float(f"l2_{i}", 1e-7, 1e-3, sampling="log")
                    ) if hp.Boolean(f"use_reg_{i}") else None
                ),
                name=f"lstm_{i}"
            )(x)

            # LayerNormalization opcional
            if hp.Boolean(f"use_layernorm_{i}"):
                x = tf.keras.layers.LayerNormalization(name=f"layernorm_{i}")(x)

        # Densa intermedia (opcional)
        if hp.Boolean("use_intermediate_dense"):
            x = tf.keras.layers.Dense(
                units=hp.Int("dense_units", 16, 128, step=16),
                activation=hp.Choice("dense_activation", ["relu", "tanh", "gelu"]),
                kernel_regularizer=(
                    tf.keras.regularizers.l2(
                        hp.Float("dense_l2", 1e-6, 1e-2, sampling="log")
                    ) if hp.Boolean("dense_use_l2") else None
                ),
                name="dense_intermediate"
            )(x)

            if hp.Boolean("use_batchnorm"):
                x = tf.keras.layers.BatchNormalization(name="batchnorm")(x)

            if hp.Boolean("use_dense_dropout"):
                x = tf.keras.layers.Dropout(
                    rate=hp.Float("dense_dropout_rate", 0.1, 0.6, step=0.1),
                    name="dense_dropout"
                )(x)

        # Capa de salida
        outputs = tf.keras.layers.Dense(
            num_classes, activation="softmax", name="output_probs"
        )(x)

        model = tf.keras.Model(inputs=inputs, outputs=outputs, name="tunable_lstm_multiclass")

        # Optimizador
        opt = hp.Choice("optimizer", ["adam", "sgd", "rmsprop"])
        if opt == "adam":
            optimizer = tf.keras.optimizers.Adam(
                learning_rate=hp.Float("adam_lr", 1e-5, 1e-3, sampling="log"),
                beta_1=hp.Float("adam_b1", 0.8, 0.99, step=0.01),
                beta_2=hp.Float("adam_b2", 0.9, 0.999, step=0.01),
                epsilon=1e-7
            )
        elif opt == "sgd":
            optimizer = tf.keras.optimizers.SGD(
                learning_rate=hp.Float("sgd_lr", 1e-4, 1e-1, sampling="log"),
                momentum=hp.Float("sgd_momentum", 0.0, 0.95, step=0.05),
                nesterov=hp.Boolean("sgd_nesterov")
            )
        else:
            optimizer = tf.keras.optimizers.RMSprop(
                learning_rate=hp.Float("rms_lr", 1e-5, 1e-3, sampling="log"),
                rho=hp.Float("rms_rho", 0.8, 0.99, step=0.01),
                momentum=hp.Float("rms_momentum", 0.0, 0.9, step=0.05),
                epsilon=1e-7
            )

        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )
        return model

    return build

Clasificacion Binaria

In [ ]:
def build_model_lstm_binary(max_length: int, vocab_size: int):
    def build(hp: kt.HyperParameters):
        # Entrada
        inputs = tf.keras.Input(
            shape=(max_length,), dtype=tf.int64, name="input_tokens"
        )

        # Embedding
        x = tf.keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=hp.Choice("embedding_dim", [64, 128]),
            name="embedding"
        )(inputs)

        # SpatialDropout (opcional)
        if hp.Boolean("use_spatial_dropout"):
            x = tf.keras.layers.SpatialDropout1D(
                rate=hp.Float("spatial_dropout_rate", 0.1, 0.5, step=0.1)
            )(x)

        # Bloque LSTM dinámico
        num_layers = hp.Int("num_lstm_layers", 1, 5)
        for i in range(num_layers):
            x = tf.keras.layers.LSTM(
                units=hp.Int(f"lstm_units_{i}", 32, 256, step=32),
                return_sequences=(i < num_layers - 1),
                recurrent_dropout=hp.Float(f"lstm_rec_dropout_{i}", 0.0, 0.5, step=0.1),
                kernel_initializer=hp.Choice(
                    f"kernel_init_{i}", ["glorot_uniform", "he_uniform", "orthogonal"]
                ),
                recurrent_initializer=hp.Choice(
                    f"recurrent_init_{i}", ["orthogonal", "glorot_uniform"]
                ),
                kernel_regularizer=(
                    tf.keras.regularizers.L1L2(
                        l1=hp.Float(f"l1_{i}", 1e-7, 1e-4, sampling="log"),
                        l2=hp.Float(f"l2_{i}", 1e-7, 1e-3, sampling="log")
                    ) if hp.Boolean(f"use_reg_{i}") else None
                ),
                name=f"lstm_{i}"
            )(x)

            # LayerNorm opcional
            if hp.Boolean(f"use_layernorm_{i}"):
                x = tf.keras.layers.LayerNormalization(name=f"layernorm_{i}")(x)

        # Densa intermedia (opcional)
        if hp.Boolean("use_intermediate_dense"):
            x = tf.keras.layers.Dense(
                units=hp.Int("dense_units", 16, 128, step=16),
                activation=hp.Choice("dense_activation", ["relu", "tanh", "gelu"]),
                kernel_regularizer=(
                    tf.keras.regularizers.l2(
                        hp.Float("dense_l2", 1e-6, 1e-2, sampling="log")
                    ) if hp.Boolean("dense_use_l2") else None
                ),
                name="dense_intermediate"
            )(x)

            if hp.Boolean("use_batchnorm"):
                x = tf.keras.layers.BatchNormalization(name="batchnorm")(x)

            if hp.Boolean("use_dense_dropout"):
                x = tf.keras.layers.Dropout(
                    rate=hp.Float("dense_dropout_rate", 0.1, 0.6, step=0.1),
                    name="dense_dropout"
                )(x)

        # Salida
        outputs = tf.keras.layers.Dense(
            1, activation="sigmoid", name="output_prob"
        )(x)

        model = tf.keras.Model(inputs=inputs, outputs=outputs, name="tunable_lstm_binary")

        # Optimizador
        opt = hp.Choice("optimizer", ["adam", "sgd", "rmsprop"])
        if opt == "adam":
            optimizer = tf.keras.optimizers.Adam(
                learning_rate=hp.Float("adam_lr", 1e-5, 1e-3, sampling="log"),
                beta_1=hp.Float("adam_b1", 0.8, 0.99, step=0.01),
                beta_2=hp.Float("adam_b2", 0.9, 0.999, step=0.01),
                epsilon=1e-7
            )
        elif opt == "sgd":
            optimizer = tf.keras.optimizers.SGD(
                learning_rate=hp.Float("sgd_lr", 1e-4, 1e-1, sampling="log"),
                momentum=hp.Float("sgd_momentum", 0.0, 0.95, step=0.05),
                nesterov=hp.Boolean("sgd_nesterov")
            )
        else:
            optimizer = tf.keras.optimizers.RMSprop(
                learning_rate=hp.Float("rms_lr", 1e-5, 1e-3, sampling="log"),
                rho=hp.Float("rms_rho", 0.8, 0.99, step=0.01),
                momentum=hp.Float("rms_momentum", 0.0, 0.9, step=0.05),
                epsilon=1e-7
            )

        model.compile(
            optimizer=optimizer,
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )
        return model

    return build

In [ ]:
def get_builder(num_classes, max_length, vocab_size):
    # Si el número de clases es 1, estamos haciendo clasificación binaria,
    # por lo tanto, se retorna la función que construye un modelo binario.
    # En caso contrario, se asume una clasificación multiclase (más de 2 clases),
    # y se retorna la función correspondiente para ese tipo de modelo.
    return build_model_lstm_binary(max_length, vocab_size) if num_classes == 1 else build_model_lstm_multiclass(max_length, num_classes, vocab_size)

In [ ]:
LSTM_EXPERIMENTS = [
    # dict(name="lstm_bin_raw",        mode="binary",     balance=False, use_weights=False),
    # dict(name="lstm_bin_balanced",   mode="binary",     balance=True,  use_weights=False),
    # dict(name="lstm_bin_weighted",   mode="binary",     balance=False, use_weights=True),

    # dict(name="lstm_multi_raw",      mode="multiclass", balance=False, use_weights=False),
    # dict(name="lstm_multi_balanced", mode="multiclass", balance=True,  use_weights=False),
    dict(name="lstm_multi_weighted", mode="multiclass", balance=False, use_weights=True),
]

In [ ]:
#Directorio raíz para almacenar los logs y modelos del tuner
LOG_ROOT = Path("tuner_logs_lstm")
LOG_ROOT.mkdir(exist_ok=True)

#Tamaño fijo de batch para entrenamiento y evaluación
BATCH_SIZE = 16

#Iteración sobre cada configuración experimental definida en LSTM_EXPERIMENTS
for cfg in tqdm(LSTM_EXPERIMENTS, desc="Configuraciones LSTM", unit="cfg"):
    print(f"\n🔎  Buscando hiperparámetros – {cfg['name']}")

    #Preparación del conjunto de datos según la configuración experimental
    train_ds, val_ds, test_ds, cw, num_classes, max_len, features = prepare_data(cfg)

    #Verificación de consistencia de los datos preparados
    sanity_check_prepare_data(train_ds, val_ds, test_ds, cw, num_classes, max_len, features)
    print()

    #Vectorización del texto (tokenización + mapeo a enteros)
    train_ds, val_ds, test_ds, vectorize_layer, vocab_size = vectorize_datasets(
        train_ds, val_ds, test_ds, features, max_len
    )

    #Directorio base para el experimento actual
    exp_dir = LOG_ROOT/ cfg["name"]

    #Constructor de modelo dinámico para binario o multiclase
    builder = get_builder(num_classes, max_len, vocab_size)

    #Configuración del tuner bayesiano
    tuner = kt.BayesianOptimization(
        hypermodel=builder,
        objective=kt.Objective("val_accuracy", direction="max"),
        max_trials=30,
        num_initial_points=10,
        seed=42,
        directory=str(exp_dir),
        project_name=cfg["name"]
    )

    #Callback de parada temprana para evitar sobreentrenamiento
    early_stopping = tf.keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True
    )

    #Callback de TensorBoard para visualizar el entrenamiento
    logs = tf.keras.callbacks.TensorBoard(
        log_dir=f"{str(exp_dir)}/tb_logs"
    )

    #Ejecución del tuner
    tuner.search(
        train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE),
        validation_data=val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE),
        epochs=50,
        class_weight=cw if cw else None,
        callbacks=[early_stopping, logs],
        verbose=2
    )

    try:
        best_model = tuner.get_best_models(1)[0]
        print("✅ Modelo cargado exitosamente desde el tuner.")
    except Exception as e:
        print(f"⚠️  No se pudo cargar el modelo desde el tuner: {e}")
        print("Reconstruyendo el modelo desde cero con los mejores hiperparámetros...")
        best_hp = tuner.get_best_hyperparameters(1)[0]
        best_model = best_builder(best_hp)
        # Si quieres, podrías entrenar aquí best_model, si aún no lo has hecho
    else:
        # Si el modelo se cargó bien, igual recupera los hiperparámetros
        best_hp = tuner.get_best_hyperparameters(1)[0]

        #Evaluación final en el conjunto de prueba
        result = best_model.evaluate(
            test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE),
            verbose=1
        )

    # Obtener etiquetas reales y predichas
    y_true = np.concatenate([y.numpy() for _, y in test_ds.batch(BATCH_SIZE)], axis=0)
    y_pred_probs = best_model.predict(test_ds.batch(BATCH_SIZE))

    # Convertir probabilidades a clases
    print(num_classes)
    if num_classes == 1:
        y_pred = (y_pred_probs > 0.5).astype("int32").flatten()
        print(f"Predicciones binarias: {np.unique(y_pred)}")
    else:
        y_pred = np.argmax(y_pred_probs, axis=1)

    #Creación del directorio del modelo
    models_dir = LOG_ROOT / cfg["name"] / "models"
    models_dir.mkdir(parents=True, exist_ok=True)

    # Evaluar y guardar resultados
    evaluate_from_predictions(y_true, y_pred, num_classes, output_dir=models_dir)

    #Guardado de resultados en test
    with open(models_dir / "test_results.txt", "w") as f:
        f.write(f"Test Loss: {result[0]:.4f} — Test Accuracy: {result[1]:.4f}\n")

    #Guardado de los mejores hiperparámetros
    with (models_dir / "best_hyperparameters.json").open("w") as f:
        json.dump(best_hp.values, f, indent=2)

    #Guardado del mejor modelo
    best_model.save(str(models_dir / f"{cfg['name']}_best_model.h5"))
    best_model.save(str(models_dir / f"{cfg['name']}_best_model.keras"))

    # Reconstrucción del modelo con los mejores hiperparámetros
    best_builder = get_builder(num_classes, max_len, vocab_size)
    best_model_rebuilt = best_builder(best_hp)

    early_stopping_2 = tf.keras.callbacks.EarlyStopping(
        patience=5,
        restore_best_weights=True
    )

    # Reentrenamiento final para obtener history
    history = best_model_rebuilt.fit(
        train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE),
        validation_data=val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE),
        epochs=100,
        class_weight=cw if cw else None,
        callbacks=[early_stopping_2],
        verbose=2
    )

    # Gráfico de las métricas de entrenamiento
    plot_training(history, models_dir / "training_plot.png")


NameError: name 'LSTM_EXPERIMENTS' is not defined

PROBAR XXL

In [ ]:
def load_best_hyperparameters(path):
    with open(path, "r") as f:
        hp_dict = json.load(f)
    hp = kt.HyperParameters()
    for key, value in hp_dict.items():
        hp.values[key] = value
    return hp

experiment_name = "lstm_multi_weighted"
suffix = "xxl"
output_dir = f"outputs/{experiment_name}_{suffix}"
output_dir = Path(output_dir)
os.makedirs(output_dir, exist_ok=True)

hp_path = "tuner_logs_lstm_4/lstm_multi_weighted/models/best_hyperparameters.json"
best_hp = load_best_hyperparameters(hp_path)

cfg = {
    "mode": "multiclass",
    "balance": False,
    "use_weights": True
}

# Prepara los datos
train_ds, val_ds, test_ds, cw, num_classes, max_len, features = prepare_data(cfg, "yelp_reviews_xxl.parquet", 3_000_000)
train_ds, val_ds, test_ds, vectorize_layer, vocab_size = vectorize_datasets(
    train_ds, val_ds, test_ds, features, max_len
)

train_ds = train_ds.batch(256).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(256).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(256).prefetch(tf.data.AUTOTUNE)

# Obtén el builder correcto y crea el modelo
builder = get_builder(num_classes, max_len, vocab_size)
model = builder(best_hp)  # <-- builder espera un objeto HyperParameters

# Crea el callback ModelCheckpoint
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=str(output_dir / "best_model.weights.h5"),
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

# Prepara fit_kwargs con acceso correcto a los hiperparámetros
fit_kwargs = {
    "epochs": 8,
    "validation_data": val_ds,
    "callbacks": [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=best_hp.values.get("early_stopping_patience", 5),
            restore_best_weights=True
        ),
        checkpoint_cb,
    ],
}
if cw:
    fit_kwargs["class_weight"] = cw

# Entrenamiento y evaluación
history = model.fit(train_ds, **fit_kwargs)

# Gráfico de las métricas de entrenamiento
plot_training(history, os.path.join(output_dir, "training_plot.png"))

results = model.evaluate(test_ds)

if num_classes == 1:
    y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
    y_pred_probs = model.predict(test_ds)
    y_pred = (y_pred_probs > 0.5).astype("int32").flatten()
else:
    y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)
    y_pred_probs = model.predict(test_ds)
    y_pred = np.argmax(y_pred_probs, axis=1)

evaluate_from_predictions(y_true, y_pred, num_classes, output_dir=output_dir)

with open(os.path.join(output_dir, "results.txt"), "w") as f:
    f.write(f"Resultados en test: {results}\n")

print("Resultados en test:", results)